## Load packages

In [ ]:
import Pkg
Pkg.activate(".") # Activate the current environment where the NeuralCrop.jl is located.

using NeuralCrop
using Lux, CUDA, LuxCUDA

import JLD2: @load, @save

CUDA.allowscalar(false)

const cdev = cpu_device()
const gdev = gpu_device()

## Initilization

In [ ]:
@load "./initial_wheat.jld2" data

climate_path = "./"
pftparameters = cft1

### time range: 2000-2019; load every 10-year climate data
all_indices = collect(1:length(data.coords))
batch_idx = all_indices;

## Simulations

In [ ]:
function run_simulation!(
    n_decades, 
    pftparameters, 
    climbuf, 
    crop, 
    crop_cal, 
    photos, 
    pet, 
    soil, 
    managed_land,
    dailyWeather, 
    output,
    InitialData,
    climate_path,
    batch_idx,
    gdev;
    wheat_model = nothing,
    ps = nothing,        
    st = nothing,      
    start_day = 1,
    end_day = 3650    
)

    ### print
    println("\n" * "="^50)
    println("🚀 start simulation")
    println("="^50)

    for i in 1:n_decades

        start_year = 2000 + (i - 1) * 10
        end_year = start_year + 9

        print(" -> simulation years: $start_year - $end_year ... ")

        file_path = joinpath(climate_path, "climate_$(start_year)_$(end_year).jld2")
        @load file_path climate

        climate_loader = ClimateDataLoader(climate, batch_idx, gdev)
        
        if i == 1
            spin_up_climbuf!(pftparameters, climate_loader.temp_spinup, climbuf, 1, gdev)
        end

        data_batch = (
            latitude = InitialData.latitude, 
            climate = climate_loader, 
            ModelState = InitialData.lpjml
        )

        daily_crop_C3!(
                start_day, end_day, 
                pftparameters, data_batch, length(batch_idx), 
                climbuf, crop, crop_cal, photos, pet, soil, 
                managed_land, dailyWeather, output, gdev
            )

        println("completed ✅")
        
    end

    println("🎉 simulation completed!")
end

run_simulation! (generic function with 1 method)

## Process-based simulations

In [ ]:
### Initilization
InitialData = InitialDataLoader(data, batch_idx, gdev)
climbuf, crop, crop_cal, photos, pet, soil, managed_land, dailyWeather, output = init_states!(pftparameters, InitialData, length(batch_idx), gdev);

### Running
run_simulation!(
    2, pftparameters, climbuf, crop, crop_cal, 
    photos, pet, soil, managed_land, dailyWeather, output, 
    InitialData, climate_path, batch_idx, gdev;
    wheat_model = wheat_model, ps = ps, st = st
)


🚀 start simulation
 -> simulation years: 2000 - 2009 ... completed ✅
 -> simulation years: 2010 - 2019 ... completed ✅
🎉 simulation completed!
